# Search Optimization

- Semantic config for language (default, korean)
- Filter & pre-filter
- Single vector search with multiple fields
    - Opt: multi-vector search
- ANN vs KNN
- Opt: Compression

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient  

In [3]:
import os

search_endpoint = os.getenv("AZSCH_ENDPOINT")  
credential = AzureKeyCredential(os.environ["AZSCH_KEY"])
print(search_endpoint)

api_key = os.environ["AZURE_OPENAI_KEY"]
azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']

https://iksch.search.windows.net


In [4]:
from azure.search.documents.models import (
    VectorizableTextQuery,
    VectorQuery,
    VectorizedQuery,
    QueryType,
    QueryCaptionType,
    QueryAnswerType,
    VectorFilterMode)

In [5]:
index_name = 'nc-articles-nochunk-index'
search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)

## semantic search

In [6]:
import time

In [9]:
def azsch_rerank_query(query, semantic_config='semantic-config', search_fields='content_vector', KNN=False, Caption=True):

    start = time.time()
    #vector_query = VectorizedQuery(vector=generate_embeddings(query), k_nearest_neighbors=3, fields="vector")
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=50, fields=search_fields, exhaustive=KNN)

    results = search_client.search(  
        search_text=query,  
        vector_queries=[vector_query],
        select=["content", "translated", "title", "article_id"],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name=semantic_config,
        query_caption=QueryCaptionType.EXTRACTIVE if Caption else None,
        #query_answer=QueryAnswerType.EXTRACTIVE,
        query_language="ko-kr",
        top=5 # for limiting text search
    ) 
    
    for result in results:  
        if result["@search.captions"]:
            caption = result["@search.captions"][0]
            print(f"{result['@search.reranker_score']:.5f}: {result['article_id']}, {result['title']} - {caption.highlights}") 
        else:
            print(f"{result['@search.reranker_score']:.5f}: {result['article_id']}, {result['title']} - {result['translated']}")  
 
    end = time.time()
    print(f"Query time: {end - start:.2f} seconds")

### Options: single fields vs multi-fields

In [10]:
azsch_rerank_query("sport bicycle")

2.75715: 755, Road-450 Red, 60 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74864: 757, Road-450 Red, 48 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74722: 754, Road-450 Red, 58 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74368: 758, Road-450 Red, 52 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</

In [11]:
azsch_rerank_query("sport bicycle", "semantic-config", "title_vector, content_vector")

2.75715: 755, Road-450 Red, 60 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74864: 757, Road-450 Red, 48 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74722: 754, Road-450 Red, 58 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74368: 758, Road-450 Red, 52 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</

In [12]:
azsch_rerank_query("sport bicycle", "semantic-config", "title_vector, content_vector", True)

2.75715: 755, Road-450 Red, 60 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74864: 757, Road-450 Red, 48 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74722: 754, Road-450 Red, 58 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</em>
2.74368: 758, Road-450 Red, 52 - A<em> true multi-sport bike </em>that offers<em> streamlined riding and a revolutionary design.</em> <em>Aerodynamic design lets you ride with the pros, </em>and the<em> gearing </em>will<em> conquer hilly roads.</

In [13]:
azsch_rerank_query("스포츠 자전거", "semantic-config", "title_vector, content_vector", True)

2.20105: 757, Road-450 Red, 48 - A<em> true multi-sport bike </em>that offers<em> streamlined riding </em>and a<em> revolutionary design.</em> <em>Aerodynamic design </em>lets you ride with the pros, and the gearing will conquer hilly roads.
2.19866: 754, Road-450 Red, 58 - A true<em> multi-sport bike </em>that offers<em> streamlined riding </em>and a<em> revolutionary design.</em> <em>Aerodynamic design </em>lets you ride with the pros, and the gearing will conquer hilly roads.
2.19821: 756, Road-450 Red, 44 - A<em> true multi-sport bike </em>that offers<em> streamlined riding </em>and a revolutionary<em> design.</em> <em>Aerodynamic design </em>lets you ride with the pros, and the gearing will conquer hilly roads.
2.19680: 755, Road-450 Red, 60 - A<em> true multi-sport bike </em>that offers<em> streamlined riding </em>and a<em> revolutionary design.</em> <em>Aerodynamic design </em>lets you ride with the pros, and the gearing will conquer<em> hilly </em>roads.
2.19289: 758, Road-450 

In [14]:
azsch_rerank_query("스포츠 자전거", "semantic-ko-config", "title_vector, content_vector", True)

2.79968: 755, Road-450 Red, 60 - <em>유선형의 라이딩과 혁신적인 디자인을</em> 제공하는<em> 진정한 멀티 스포츠 자전거입</em>니다. <em>공기역학적 디자인으로 프로와 함께 라이딩할 수 </em>있으며<em> 기어링은 언덕이 많은 도로를</em> 정<em>복할</em> 수 있습니다.
2.77841: 757, Road-450 Red, 48 - <em>유선형의 라이딩과 혁신적인 디자인을</em> 제공하는<em> 진정한</em><em> 멀티 스포츠 자전거입</em>니다. <em>공기역학적 디자인으로 프로와 함께 라이딩할 수 </em>있으며<em> 기어링은 언덕이 많은 도로를</em> 정복할 수 있습니다.
2.76353: 758, Road-450 Red, 52 - <em>유선형의 라이딩과 혁신적인 디자인을</em> 제공하는<em> 진정한</em><em> 멀티 스포츠 자전거입</em>니다. <em>공기역학적 디자인으로 프로와 함께 라이딩할 수 </em>있으며<em> 기어링은 언덕이 많은 도로를</em> 정복할 수 있습니다.
2.75006: 756, Road-450 Red, 44 - <em>유선형의 라이딩과 혁신적인 디자인을</em> 제공하는<em> 진정한</em><em> 멀티 스포츠 자전거입</em>니다. <em>공기역학적 디자인으로 프로와 함께 라이딩할 수 </em>있으며<em> 기어링은</em><em> 언덕이</em> 많은<em> 도로를</em> 정복할 수 있습니다.
2.73446: 754, Road-450 Red, 58 - <em>유선형의 라이딩과 혁신적인 디자인을</em> 제공하는<em> 진정한</em><em> 멀티 스포츠 자전거입</em>니다. <em>공기역학적 디자인으로 프로와 함께 라이딩할 수 </em>있으며<em> 기어링은</em><em> 언덕이</em> 많은<em> 도로를</em> 정복할 수 있습니다.
Query time: 1.39 seconds


In [15]:
azsch_rerank_query("스포츠 자전거", "semantic-ko-config", "title_vector, content_vector", True, False)

2.79968: 755, Road-450 Red, 60 - 유선형의 라이딩과 혁신적인 디자인을 제공하는 진정한 멀티 스포츠 자전거입니다. 공기역학적 디자인으로 프로와 함께 라이딩할 수 있으며 기어링은 언덕이 많은 도로를 정복할 수 있습니다.
2.77841: 757, Road-450 Red, 48 - 유선형의 라이딩과 혁신적인 디자인을 제공하는 진정한 멀티 스포츠 자전거입니다. 공기역학적 디자인으로 프로와 함께 라이딩할 수 있으며 기어링은 언덕이 많은 도로를 정복할 수 있습니다.
2.76353: 758, Road-450 Red, 52 - 유선형의 라이딩과 혁신적인 디자인을 제공하는 진정한 멀티 스포츠 자전거입니다. 공기역학적 디자인으로 프로와 함께 라이딩할 수 있으며 기어링은 언덕이 많은 도로를 정복할 수 있습니다.
2.75006: 756, Road-450 Red, 44 - 유선형의 라이딩과 혁신적인 디자인을 제공하는 진정한 멀티 스포츠 자전거입니다. 공기역학적 디자인으로 프로와 함께 라이딩할 수 있으며 기어링은 언덕이 많은 도로를 정복할 수 있습니다.
2.73446: 754, Road-450 Red, 58 - 유선형의 라이딩과 혁신적인 디자인을 제공하는 진정한 멀티 스포츠 자전거입니다. 공기역학적 디자인으로 프로와 함께 라이딩할 수 있으며 기어링은 언덕이 많은 도로를 정복할 수 있습니다.
Query time: 1.25 seconds


### Options: Filter

In [18]:
def azsch_rerank_query_filter(query, condition):
    start = time.time()
    #vector_query = VectorizedQuery(vector=generate_embeddings(query), k_nearest_neighbors=3, fields="vector")
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=50, fields="title_vector, content_vector", exhaustive=True)

    results = search_client.search(  
        search_text=query,  
        vector_queries=[vector_query],
        select=["content", "translated", "title", "article_id"],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name='semantic-ko-config',
        query_caption=QueryCaptionType.EXTRACTIVE,
        #query_answer=QueryAnswerType.EXTRACTIVE,
        query_language="ko-kr",
        vector_filter_mode=VectorFilterMode.PRE_FILTER,
        filter=f"board_id gt {condition}",
        top=5 # for limiting text search
    ) 

    for result in results:  
        if result["@search.captions"]:
            caption = result["@search.captions"][0]
            print(f"{result['@search.reranker_score']:.5f}: {result['article_id']}, {result['title']} - {caption.highlights}") 
        else:
            print(f"{result['@search.reranker_score']:.5f}: {result['article_id']}, {result['title']} - {result['translated']}")   
        end = time.time()
    print(f"Query time: {end - start:.2f} seconds")
 

In [19]:
azsch_rerank_query_filter("스포츠 자전거", 20)

2.21390: 876, Hitch Rack - 4-Bike - <em>4대의 자전거를</em><em> 안전하게 운반할</em> 수 있습니다. <em>강철 구조, 2" 수신기 히치에 맞습니다.</em>
2.08525: 878, Fender Set - Mountain - <em>클립온 펜더는</em><em> 대부분의 산악 자전거에</em><em> 적합합</em>니다.
2.02747: 829, Touring Rear Wheel - <em>뛰어난 공기역학적 림은 부드러운 승차감을 보장합니다.</em>
1.99096: 827, ML Road Rear Wheel - <em>스테인레스 스틸 스포크가있는 알루미늄 합금 림;</em><em> 빠른 속도를 위해 제작되</em>었습니다.
1.96934: 881, Short-Sleeve Classic Jersey, S - <em>짧은 소매의</em><em> 클래식한 통기성 저지로 수분 조절 기능이 뛰어나고, 앞면 지퍼와 뒷면 포켓 3개가</em> 있습니다.
Query time: 1.30 seconds
